# Version 2
New features
- Block structure for SimpleCNN() class definition via CNNBlock()
- Switched from FC layers at end to a single CNN with out_channel=1

## Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data prepping

In [ ]:
# Define data paths
data_paths = ['./data']  # TODO: Update with actual data paths

In [ ]:
# Create datasets (if applicable)
## Inspect data
# Get sample and label from 0th index
# Print shapes and types
# Print labels and PV locations

In [ ]:
# Visualize sample and corresponding label (create in utils)

In [ ]:
# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

## Create the model
This will be the simplest form of the model possible, no bells and whistles, 
completely hardcoded.

In [ ]:
class CNNBlock(nn.Module):
    """
    Defines a single convolutional block for a CNN.

    This block consists of a convolutional layer, batch normalization,
    a ReLU activation, and a max-pooling layer, bundled as a sequential module.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3):
        """
        Initializes the layers of the CNNBlock.

        Args:
            in_channels (int): Number of channels in the input image.
            out_channels (int): Number of channels produced by the convolution.
            kernel_size (int, optional): Size of the convolving kernel. Defaults to 3.
            padding (int, optional): Zero-padding added to both sides of the input. Defaults to 1.
        """
        super(CNNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                      padding=(kernel_size-1)//2),  # same padding,
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU()  # Could start with ReLU instead and switch to LeakyReLU later
            # Can't do MaxPool1d because we need equal sized input and output
            # nn.MaxPool1d(kernel_size=2, stride=2)
        )
    
    def forward(self, x):
        """
        Defines the forward pass of the CNNBlock.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_channels, length).

        Returns:
            torch.Tensor: Output tensor after applying the convolutional block.
        """
        return self.block(x)

In [ ]:
# Verify the CNNBlock
print("--- Verifying CNNBlock ---\n")

# Instantiate the block with 1 input channel and 1 output channel
verify_cnn_block = CNNBlock(in_channels=1, out_channels=1)
print("Block Structure:\n")
print(verify_cnn_block)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=1, channels=1, length=4000), i.e.,
# a single 4000 bin histogram
dummy_input = torch.randn(1, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the block
output = verify_cnn_block(dummy_input)
print(f"Output tensor shape: {output.shape}")

In [ ]:
# Assemble the full model
class SimpleCNN(nn.Module):
    def __init__(self):
        # Initialize parent nn.Module class
        super(SimpleCNN, self).__init__()

        self.conv1_block = CNNBlock(1, 16)
        self.conv2_block = CNNBlock(16, 32)
        self.conv3_block = CNNBlock(32, 64)
        self.conv4_block = CNNBlock(64, 64)
        self.conv5_block = CNNBlock(64, 64)

        self.head = nn.Conv1d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.conv1_block(x)
        x = self.conv2_block(x)
        x = self.conv3_block(x)
        x = self.conv4_block(x)
        x = self.conv5_block(x)

        # Pass through the locator head
        x = self.head(x)  # (batch_size, 1, 4000)
        return x.squeeze(1)  # (batch_size, 4000)


In [ ]:
# Verify the SimpleCNN
print("--- Verifying SimpleCNN ---\n")

# Verify the structure of the model
# Instantiate the model with 15 output classes
verify_simple_cnn = SimpleCNN()
print("Model Structure:\n")
print(verify_simple_cnn)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=64, channels=1, length=4000)
dummy_input = torch.randn(64, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the model
output = verify_simple_cnn(dummy_input)
print(f"Output tensor shape: {output.shape}")

In [ ]:
# Create model
model = SimpleCNN()

# Loss function
# TODO: understand why we use a custom cost function instead of a standard one
loss_function = None

# Optimizer
# TODO: weight_decay=5e-4
# TODO: learning rate scheduler (e.g., StepLR, ReduceLROnPlateau, 
# CosineAnnealingLR)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# TODO: untangle mess model loops scattered all throughout pv-finder, in loss.py
# / alt_loss_A.py, training.py, utilities.py (progress bar, etc.), device
# allocation, epoch loop, train loop, validation loop
#
# I will sketch out the training loop below, but it will need to be refactored
# for pv-finder's specific needs and to ensure clean separation of concerns.

In [ ]:
# Define training loop for one epoch (one pass through the training data)
def train_epoch(model, loss_function, optimizer, train_loader, device):
    model.to(device)  # Move model to the appropriate device
    model.train()  # Set model to training mode
    total_loss = 0
    n_correct_predictions, n_total_predictions = 0, 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to the appropriate device
        inputs, targets = inputs.to(device), targets.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = loss_function(outputs, targets)

        # Backpropagation to compute gradients
        loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate total loss
        total_loss += loss.item()

        # Calculate efficiency metrics
        # TODO: Understand Simon's implementation

        # Count correct predictions
        # TODO: This is a placeholder for calculating correct predictions
        predicted_indices = outputs.argmax(dim=1)
        correct_predictions = predicted_indices.eq(targets)
        n_correct_in_batch = correct_predictions.sum().item()
        # Accumulate total correct predictions
        n_correct_predictions += n_correct_in_batch

        # Accumulate total predictions
        batch_size = len(targets)
        n_total_predictions += batch_size

    accuracy = n_correct_predictions / n_total_predictions
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch loss (avg.): {avg_loss:.3f} | Accuracy: {accuracy*100:.2f}%")

    return model, avg_loss

In [ ]:
# Define one validation loop
def evaluate(model, val_loader, device):
    model.eval()  # Set model to evaluation mode
    n_correct_predictions, n_total_predictions = 0, 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            # Count correct predictions
            predicted_indices = outputs.argmax(dim=1)
            correct_predictions = predicted_indices.eq(targets)
            n_correct_in_batch = correct_predictions.sum().item()
            n_correct_predictions += n_correct_in_batch

            # Count total predictions
            batch_size = len(targets)
            n_total_predictions += batch_size
        
        # Calculate accuracy
        accuracy = n_correct_predictions / n_total_predictions
        print(f"Validation accuracy: {accuracy*100:.2f}%")
    return accuracy

In [ ]:
# Train and evaluate the model for a number of epochs
def train_and_evaluate(model, loss_function, optimizer, train_loader, val_loader, device, n_epochs):
    for epoch in range(n_epochs):
        print(f"Epoch {epoch + 1}/{n_epochs}")
        trained_model, train_loss = train_epoch(model, loss_function, optimizer, train_loader, device)
        accuracy = evaluate(model, val_loader, device)
    return trained_model, accuracy


In [ ]:
n_epochs = 100  # TODO: placeholder value
trained_model, _ = train_and_evaluate(model=model, 
                                   loss_function=loss_function, 
                                   optimizer=optimizer, 
                                   train_loader=train_loader, 
                                   val_loader=val_loader, 
                                   device=device, 
                                   n_epochs=n_epochs)